# Lesson 8, one call at a time

Lesson 8 grouped 2,000 customers with k-means, watched k-means and a tree of merges
fail on a core of bots inside a ring of humans where density-based spatial
clustering of applications with noise (DBSCAN) succeeded, and caught
fraudulent accounts by how badly principal component analysis (PCA) rebuilds them.
This notebook takes each step on its own, on a few points where a pencil can follow
it and on the lesson's own data where the slides' numbers come from, so that the
slide's number and the call that makes it sit side by side.

**How to use it.** Each section is one *atom*. Before each code cell that teaches
something there is a question under *Your prediction, before you run the next cell*:
answer it first, then run. `Consolidation/00_the_scikit_learn_api.ipynb` explains
`fit`, `predict`, `transform` and `random_state`; this one assumes them.

Every atom names the slide it comes from, by title, and lesson 8's three notebooks
treat each idea at full scale. WCSS stands for the within-cluster sum of squares,
ARI for the adjusted Rand index. The whole notebook runs in about a minute.

## Setup

In [1]:
# Everything used here ships with the course image:
# pip install numpy==1.26.4 pandas==2.1.1 scipy==1.11.3 scikit-learn==1.3.1

from itertools import combinations

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Lesson 8's three datasets, from the generator beside this notebook. The true
# groups and the anomaly flags are kept aside: they are used only to check.
from retail_data import make_bot_sessions, make_customer_features, make_customer_segments

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 120)

customers = make_customer_segments()
X_cust = customers[["annual_spend_eur", "visits_per_month"]].values
Xs_cust = StandardScaler().fit_transform(X_cust)

sessions = make_bot_sessions()
Xs_sess = StandardScaler().fit_transform(sessions[["session_duration_min", "pages_viewed"]].values)
is_bot = (sessions["true_group"] == "bot").values

accounts = make_customer_features()
columns = [c for c in accounts.columns if c != "is_anomaly"]
Xs_acc = StandardScaler().fit_transform(accounts[columns].values)
is_anomaly = accounts["is_anomaly"].values

print(f"{len(customers)} customers, {len(sessions)} sessions, {len(accounts)} accounts x {len(columns)} columns")

2000 customers, 1500 sessions, 2000 accounts x 8 columns


## 1. Why standardise before measuring distance

k-means gives each customer to the nearest centre, and *nearest* is a straight-line distance. Measure it between two customers in euros and visits, then after `StandardScaler`.

*From:* slides *One centre per group*; *2,000 customers, no labels*; handout section 2.1.

### Your prediction, before you run the next cell

Customers 0 and 2 differ by about 780 EUR a year and 5 visits a month. In raw units, what share of their squared distance comes from spend? After standardising?

In [2]:
a, b = 0, 2
raw = X_cust[a] - X_cust[b]
std = Xs_cust[a] - Xs_cust[b]
print("raw difference (EUR, visits):     ", raw.round(2))
print(f"share of the squared distance from spend, raw:          {raw[0]**2 / (raw**2).sum():.4%}")
print("standardised difference:           ", std.round(3))
print(f"share of the squared distance from spend, standardised: {std[0]**2 / (std**2).sum():.1%}")
print("\nstandard deviations used:", X_cust.std(axis=0).round(2))

raw difference (EUR, visits):      [781.38   4.72]
share of the squared distance from spend, raw:          99.9963%
standardised difference:            [1.948 1.974]
share of the squared distance from spend, standardised: 49.3%

standard deviations used: [401.14   2.39]


**What to look at.** In euros and visits the two customers differ by 781.38 and 4.72, and spend makes 99.9963% of their squared distance: visits per month count for nothing, only because their numbers are small. After `StandardScaler` each column is divided by its standard deviation, 401.14 EUR and 2.39 visits, and the two differences become 1.948 and 1.974: spend now makes 49.3%, and both columns have a say.

**Try changing.** run `KMeans(n_clusters=4, n_init=10, random_state=0)` on the raw `X_cust` and compare its `labels_` with the true segments by `adjusted_rand_score`.

## 2. Six customers by hand, against `KMeans` from the same start

Two moves, repeated: give each customer to the nearest centre, move each centre to the mean of its customers. Start on A and B, print the WCSS after every move, then let `KMeans` run from the same start.

*From:* slides *Two moves, repeated*; *The within-cluster sum of squares, as a formula*; *Six customers, by hand*; handout section 2.3.

### Your prediction, before you run the next cell

The start puts both centres inside the small group. Will the WCSS ever go up from one step to the next? Where will B end up?

In [3]:
six = np.array([[1, 1], [1, 2], [2, 1], [8, 6], [9, 6], [8, 7]], dtype=float)   # A..F
names = np.array(list("ABCDEF"))
centres = six[[0, 1]].copy()                                   # start on A and B

def wcss(X, labels, centres):
    return ((X - centres[labels]) ** 2).sum()

for round_ in (1, 2, 3):
    labels = ((six[:, None] - centres[None]) ** 2).sum(axis=2).argmin(axis=1)   # assign
    print(f"round {round_} assign: groups {''.join(names[labels == 0])} / {''.join(names[labels == 1])}"
          f"   WCSS {wcss(six, labels, centres):.4f}")
    centres = np.array([six[labels == j].mean(axis=0) for j in (0, 1)])         # move
    print(f"round {round_} move:   centres {centres.round(4).tolist()}   WCSS {wcss(six, labels, centres):.4f}")

km = KMeans(n_clusters=2, init=six[[0, 1]], n_init=1).fit(six)
print(f"\nKMeans from the same start: inertia_ {km.inertia_:.4f}, centres {km.cluster_centers_.round(4).tolist()}")

round 1 assign: groups AC / BDEF   WCSS 220.0000
round 1 move:   centres [[1.5, 1.0], [6.5, 5.25]]   WCSS 56.2500
round 2 assign: groups ABC / DEF   WCSS 16.6875
round 2 move:   centres [[1.3333, 1.3333], [8.3333, 6.3333]]   WCSS 2.6667
round 3 assign: groups ABC / DEF   WCSS 2.6667
round 3 move:   centres [[1.3333, 1.3333], [8.3333, 6.3333]]   WCSS 2.6667

KMeans from the same start: inertia_ 2.6667, centres [[1.3333, 1.3333], [8.3333, 6.3333]]


**What to look at.** The WCSS goes 220, 56.25, 16.6875, 2.6667 and then stays: it never goes up, because each assign and each move can only shorten the lines. B starts as a centre and still leaves its own group in round 2, when it is nearer the centre of A and C (1.5, 1) than the centre of B, D, E, F (6.5, 5.25). `KMeans` from the same start ends at the same centres and an `inertia_` of 2.6667, the hand-computed WCSS.

**Try changing.** start on D and E, `six[[3, 4]]`: how many rounds, and which final WCSS?

## 3. The mean is the best centre, in squared distance

The move step puts each centre at the mean of its group. Try other centres for the group ABC and compare the sum of squared distances.

*From:* slide *Why it always stops*; handout section 2.4.

### Your prediction, before you run the next cell

Will the median, or a point shifted a little from the mean, ever give a smaller sum of squares than the mean?

In [4]:
abc = six[:3]
def sum_sq(centre):
    return ((abc - centre) ** 2).sum()

mean = abc.mean(axis=0)
for name, centre in [("mean", mean), ("median", np.median(abc, axis=0)),
                     ("mean + (0.1, 0)", mean + [0.1, 0]), ("mean - (0, 0.3)", mean - [0, 0.3]),
                     ("customer A", abc[0])]:
    print(f"{name:<16} {np.round(centre, 4)}  sum of squared distances {sum_sq(centre):.4f}")

mean             [1.3333 1.3333]  sum of squared distances 1.3333
median           [1. 1.]  sum of squared distances 2.0000
mean + (0.1, 0)  [1.4333 1.3333]  sum of squared distances 1.3633
mean - (0, 0.3)  [1.3333 1.0333]  sum of squared distances 1.6033
customer A       [1. 1.]  sum of squared distances 2.0000


**What to look at.** The mean (1.3333, 1.3333) gives 1.3333, the smallest of the five. Moving it 0.1 to the right gives 1.3633, 0.3 down gives 1.6033, and the median, which here is customer A, gives 2.0. That is why a move step can never raise the WCSS: for squared distance the mean is the best centre of a group.

**Try changing.** sum *plain* distances, `np.linalg.norm(abc - centre, axis=1).sum()`, instead of squared ones: is the mean still the best of the five?

## 4. Same data, two starts, and the remedy

The same k-means from two different random picks of four customers, on the 2,000 standardised customers; then `KMeans` with one start and with ten.

*From:* slides *Same data, two starts*; *The remedy: several starts, keep the best*; handout section 2.4.

### Your prediction, before you run the next cell

Will every random start end at the same WCSS? If not, which of the two, `n_init=1` or `n_init=10`, can you trust without checking?

In [5]:
def kmeans_by_hand(X, centres, max_rounds=100):
    for _ in range(max_rounds):
        labels = ((X[:, None] - centres[None]) ** 2).sum(axis=2).argmin(axis=1)
        new = np.array([X[labels == j].mean(axis=0) for j in range(len(centres))])
        if np.allclose(new, centres):
            break
        centres = new
    return labels, centres

for seed in (1, 28):
    pick = np.random.default_rng(seed).choice(len(Xs_cust), 4, replace=False)
    labels, centres = kmeans_by_hand(Xs_cust, Xs_cust[pick])
    print(f"start {seed:>2}: WCSS {wcss(Xs_cust, labels, centres):.1f}")

one = [KMeans(n_clusters=4, init="random", n_init=1, random_state=s).fit(Xs_cust).inertia_ for s in range(30)]
print(f"\nKMeans, one random start, 30 seeds: best {min(one):.1f}, worst {max(one):.1f}")
print(f"KMeans, n_init=10 (k-means++ starts): {KMeans(n_clusters=4, n_init=10, random_state=0).fit(Xs_cust).inertia_:.4f}")

start  1: WCSS 374.1
start 28: WCSS 1390.4

KMeans, one random start, 30 seeds: best 374.1, worst 1391.2


KMeans, n_init=10 (k-means++ starts): 374.0919


**What to look at.** The same ten lines from two starts: 374.1 against 1,390.4. Over 30 seeds, `KMeans` with one random start goes from 374.1 to 1,391.2, and nothing in a single run says which kind of run it was. With `n_init=10` and k-means++ starts it reports 374.0919: the best of ten, which is the remedy. The notebooks always pass `n_init=10` explicitly, because scikit-learn 1.4 and later default to a single start.

**Try changing.** count how many of the 30 one-start runs end above 1,000.

## 5. The best WCSS can only fall as k grows

Fit `KMeans` for k from 1 to 8 and read `inertia_`. With k = 1 the one centre is the mean, so the WCSS is the total sum of squares: for standardised data, the number of customers times the number of columns.

*From:* slides *How many groups?*; *Elbow and silhouette, on 2,000 customers*; handout section 2.5.

### Your prediction, before you run the next cell

What is the WCSS at k = 1, before running anything? Which k has the lowest WCSS, and does that make it the right k?

In [6]:
for k in range(1, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs_cust)
    print(f"k = {k}: WCSS {km.inertia_:8.2f}")
print(f"\nm x n = {Xs_cust.shape[0]} x {Xs_cust.shape[1]} = {Xs_cust.size}")

k = 1: WCSS  4000.00


k = 2: WCSS  2128.53
k = 3: WCSS  1125.76
k = 4: WCSS   374.09
k = 5: WCSS   320.60


k = 6: WCSS   286.49


k = 7: WCSS   244.46
k = 8: WCSS   210.37

m x n = 2000 x 2 = 4000


**What to look at.** At k = 1 the WCSS is 4000.00, exactly m x n = 2000 x 2: one centre at the mean, and each standardised column contributes a variance of 1 per customer. From there the best WCSS only falls, 2128.53, 1125.76, 374.09 and on to 210.37 at k = 8, so "lowest WCSS" would always pick the largest k tried. What matters is the bend: the fall from 3 to 4 is about 752, from 4 to 5 only about 53.

**Try changing.** add k = 20 and k = 50 to the loop: does the WCSS ever rise?

## 6. The silhouette of customer B, by hand

For B in the grouping ABC / DEF: $a$ is the mean distance to the other members of its own group, $b$ the mean distance to the members of the other group, and $s = (b - a) / \max(a, b)$. No centre appears anywhere.

*From:* slides *The silhouette of one point*; *The silhouette, as a formula*; handout section 2.5.

### Your prediction, before you run the next cell

Will the silhouette of B be close to 1, 0 or -1? Would it change if you used the distances from B to the two *centres* instead?

In [7]:
labels6 = np.array([0, 0, 0, 1, 1, 1])
d = np.linalg.norm(six[:, None] - six[None], axis=2)      # every distance between two customers
a = d[1, [0, 2]].mean()                                   # B to A and C
b = d[1, [3, 4, 5]].mean()                                # B to D, E, F
print(f"a = mean({d[1, [0, 2]].round(3).tolist()}) = {a:.3f}")
print(f"b = mean({d[1, [3, 4, 5]].round(3).tolist()}) = {b:.3f}")
print(f"s = (b - a) / max(a, b) = {(b - a) / max(a, b):.3f}")
print(f"silhouette_samples for B: {silhouette_samples(six, labels6)[1]:.3f}")

c0, c1 = six[:3].mean(axis=0), six[3:].mean(axis=0)
a_c, b_c = np.linalg.norm(six[1] - c0), np.linalg.norm(six[1] - c1)
print(f"\nthe slip, with centres: ({b_c:.3f} - {a_c:.3f}) / {max(a_c, b_c):.3f} = {(b_c - a_c) / max(a_c, b_c):.3f}")

a = mean([1.0, 1.414]) = 1.207
b = mean([8.062, 8.944, 8.602]) = 8.536
s = (b - a) / max(a, b) = 0.859
silhouette_samples for B: 0.859

the slip, with centres: (8.518 - 0.745) / 8.518 = 0.912


**What to look at.** a = (1 + 1.414) / 2 = 1.207, b = (8.062 + 8.944 + 8.602) / 3 = 8.536, and s = (8.536 - 1.207) / 8.536 = 0.859, the same as `silhouette_samples`. B sits well inside its group. The slip, using the distances from B to the two centres, gives 0.912: a different number, because the silhouette is built from distances to other *points*, never to centres. That is also why it can score a grouping that has no centres, like DBSCAN's.

**Try changing.** compute the silhouette of customer D the same way, and check it against `silhouette_samples(six, labels6)[3]`.

## 7. The adjusted Rand index counts pairs, not names

Compare k-means' four clusters with the true segments. Rename the clusters at random: the ARI must not move, because it only asks, for every pair of customers, whether both groupings put them together or apart.

*From:* slides *Checking against the truth: the adjusted Rand index*; *k-means against the truth*; handout section 2.6.

### Your prediction, before you run the next cell

Cluster 0 is not "window shoppers". Does the ARI care? On how many of the 1,999,000 pairs of customers do the two groupings agree?

In [8]:
km4 = KMeans(n_clusters=4, n_init=10, random_state=0).fit(Xs_cust)
truth = customers["true_segment"].values
print(f"ARI: {adjusted_rand_score(truth, km4.labels_):.4f}")

renamed = np.random.default_rng(0).permutation(4)[km4.labels_]      # same groups, other names
print(f"ARI after renaming the clusters: {adjusted_rand_score(truth, renamed):.4f}")

same_truth = truth[:, None] == truth[None]
same_km = km4.labels_[:, None] == km4.labels_[None]
upper = np.triu(np.ones_like(same_truth, dtype=bool), k=1)           # each pair once
agree = (same_truth == same_km)[upper]
print(f"pairs: {agree.size:,}; the two groupings agree on {agree.sum():,} ({agree.mean():.2%}) - the plain Rand index")

ARI: 0.9850
ARI after renaming the clusters: 0.9850


pairs: 1,999,000; the two groupings agree on 1,987,657 (99.43%) - the plain Rand index


**What to look at.** The ARI is 0.9850 before and after renaming the clusters: it never compares names, only pairs. Out of 1,999,000 pairs of customers, the two groupings agree on 1,987,657, 99.43%, together in both or apart in both. That share is the plain Rand index; the adjusted version rescales it so that a random grouping scores 0.

**Try changing.** compare the truth with a random grouping, `np.random.default_rng(1).integers(0, 4, len(truth))`: what are the plain agreement and the ARI?

## 8. Why k-means must fail on a core inside a ring

With two centres, every session goes to the nearer one, so the border between the two groups is a straight line: the set of points equally far from both centres. Check it on the sessions.

*From:* slides *1,500 sessions, no labels*; *Why k-means must fail here*; handout sections 3.1 and 3.2.

### Your prediction, before you run the next cell

Can any straight line put the dense core on one side and the whole ring around it on the other? What ARI do you expect?

In [9]:
km2 = KMeans(n_clusters=2, n_init=10, random_state=0).fit(Xs_sess)
c0, c1 = km2.cluster_centers_
side = Xs_sess @ (c1 - c0) - (c1 @ c1 - c0 @ c0) / 2      # > 0 exactly when nearer to centre 1
print(f"sessions whose side of the line matches the k-means label: {((side > 0) == (km2.labels_ == 1)).mean():.1%}")
print(f"ARI against the truth: {adjusted_rand_score(is_bot, km2.labels_):+.3f}")
print(pd.crosstab(np.where(is_bot, "bot", "human"), km2.labels_, rownames=["truth"], colnames=["k-means cluster"]))

sessions whose side of the line matches the k-means label: 100.0%
ARI against the truth: -0.046


k-means cluster    0    1
truth                    
bot                0  180
human            540  780


**What to look at.** For 100.0% of the sessions the k-means label is exactly the side of one straight line, the line of points equally far from the two centres. No straight line can put a core on one side and the ring around it on the other, so k-means cuts the picture in two halves: all 180 bots fall in one cluster together with 780 humans, and the ARI is -0.046, no better than chance.

**Try changing.** run `KMeans` with `n_init=50`: does any start find the core and the ring?

## 9. Ward's merges on eight points

Hierarchical clustering merges the two closest groups, again and again; Ward linkage calls closest the merge that adds least to the WCSS. Read the six merge heights from `linkage`, and check the first against the plain distance.

*From:* slides *Hierarchical clustering: a tree of merges*; *Eight points, five merges, one cut*; *Ward on the sessions: the same failure*; handout section 3.3.

### Your prediction, before you run the next cell

The first merge joins two single points. Will its height be their distance? Where will the heights jump?

In [10]:
eight = np.array([[1.0, 1.0], [1.6, 1.3], [1.2, 2.3], [4.2, 4.0],
                  [5.0, 4.4], [4.5, 5.3], [8.2, 1.6], [8.8, 2.6]])   # A..H
merges = linkage(eight, method="ward")
print("merge heights:", merges[:, 2].round(3))
print(f"distance A-B: sqrt(0.6^2 + 0.3^2) = {np.sqrt(0.6**2 + 0.3**2):.3f}")

print("\ncut into 3 groups:", AgglomerativeClustering(n_clusters=3).fit(eight).labels_)
ward = AgglomerativeClustering(n_clusters=2).fit(Xs_sess)
print(f"Ward on the 1,500 sessions, cut at 2: ARI {adjusted_rand_score(is_bot, ward.labels_):+.3f}")

merge heights: [ 0.671  0.894  1.166  1.275  1.333  7.193 10.236]
distance A-B: sqrt(0.6^2 + 0.3^2) = 0.671

cut into 3 groups: [0 0 0 1 1 1 2 2]
Ward on the 1,500 sessions, cut at 2: ARI -0.062


**What to look at.** The first merge, A with B, happens at 0.671, which is their plain distance: for two single points, Ward's height is the distance. The next four stay below 1.4, and then the height jumps to 7.193: the long stretch with no merge is the tree saying "three groups", and the cut at three gives ABC, DEF, GH. On the 1,500 sessions Ward, cut at 2, scores -0.062: it rewards compact groups, the same idea as k-means, and fails the same way.

**Try changing.** `linkage(eight, method="single")`: are the heights different? Does the cut at three change?

## 10. Core, border and noise, counted by hand

Twelve points, `eps = 0.9`, `min_samples = 4`. A core point has at least 4 points in its disc, itself included; a border point is not core but lies in a core point's disc; noise is neither. Count, then ask `DBSCAN`.

*From:* slides *DBSCAN: is this point in a crowded place?*; *Core, border and noise, drawn*; handout section 3.4.

### Your prediction, before you run the next cell

How many of each kind? Does a border point make its own neighbours join the group?

In [11]:
twelve = np.array([[2.0, 2.0], [2.6, 2.3], [2.2, 2.9], [3.0, 2.8], [2.7, 3.5], [3.5, 3.4],
                   [1.6, 2.7], [3.3, 2.2], [4.4, 3.9], [5.3, 4.5], [6.2, 1.4], [1.0, 5.6]])
in_disc = np.linalg.norm(twelve[:, None] - twelve[None], axis=2) <= 0.9
count = in_disc.sum(axis=1)                       # itself included
core = count >= 4
border = ~core & (in_disc & core[None]).any(axis=1)
noise = ~core & ~border
print("points in each disc:", count)
print(f"by hand: {core.sum()} core, {border.sum()} border, {noise.sum()} noise")

db = DBSCAN(eps=0.9, min_samples=4).fit(twelve)
print("DBSCAN labels:     ", db.labels_)
print("DBSCAN core points:", db.core_sample_indices_, " by hand:", np.flatnonzero(core))

points in each disc: [3 5 5 6 4 3 3 3 1 1 1 1]
by hand: 4 core, 4 border, 4 noise
DBSCAN labels:      [ 0  0  0  0  0  0  0  0 -1 -1 -1 -1]
DBSCAN core points: [1 2 3 4]  by hand: [1 2 3 4]


**What to look at.** The disc counts, the point itself included, are 3 5 5 6 4 3 3 3 1 1 1 1. Four points have at least 4, the core points 1, 2, 3, 4, exactly `core_sample_indices_`. Four more are not core but lie in a core point's disc: border, inside the group. The last four are neither: noise, label -1. A border point joins the group but cannot extend it, because only core points pass the group on.

**Try changing.** set `min_samples=3`: which points become core, and does the group grow?

## 11. The k-distance plot: the point counts itself, and 65 is not 13

For each session, the distance to its 10th nearest neighbour, counting the session itself as the first, as `min_samples=10` does. Count how many lie beyond 0.30, then run DBSCAN and count the noise.

*From:* slides *Choosing eps: the k-distance plot*; *The bend in the curve*; *DBSCAN on the sessions*; *The 13 noise sessions*; handout sections 3.5 and 3.6.

### Your prediction, before you run the next cell

If 65 sessions have their 10th neighbour farther than 0.30, how many will DBSCAN call noise with `eps = 0.30`: 65, more, or fewer?

In [12]:
dist, idx = NearestNeighbors(n_neighbors=10).fit(Xs_sess).kneighbors(Xs_sess)
print("first neighbour of the first three sessions:", idx[:3, 0], "(each is itself, at distance", dist[:3, 0], ")")
tenth = dist[:, 9]
beyond = tenth > 0.30
print(f"sessions whose 10th neighbour is beyond 0.30: {beyond.sum()}")

db = DBSCAN(eps=0.30, min_samples=10).fit(Xs_sess)
core = np.zeros(len(Xs_sess), dtype=bool); core[db.core_sample_indices_] = True
noise = db.labels_ == -1
print(f"not core: {(~core).sum()}   of which border: {(~core & ~noise).sum()}   noise: {noise.sum()}")
print(f"noise sessions that are humans: {(noise & ~is_bot).sum()} of {noise.sum()};   ARI {adjusted_rand_score(is_bot, db.labels_):.3f}")

first neighbour of the first three sessions: [0 1 2] (each is itself, at distance [0. 0. 0.] )
sessions whose 10th neighbour is beyond 0.30: 65
not core: 65   of which border: 52   noise: 13
noise sessions that are humans: 13 of 13;   ARI 0.941


**What to look at.** Each session's first neighbour is itself, at distance 0, so the 10th column is the 9th other session, matching DBSCAN, which counts the point itself in `min_samples`. 65 sessions have their 10th neighbour beyond 0.30, so 65 are not core; but 52 of them lie inside a core session's disc and join a group as border points. Only 13 are noise, all of them humans, and the ARI is 0.941.

**Try changing.** `eps=0.20` and `eps=0.60`: how many noise sessions and how many groups?

## 12. The silhouette prefers the wrong answer, on the same sessions

The silhouette has no place for a noise point, so DBSCAN's is computed on the sessions it placed. Score k-means on the same sessions too, so the comparison is like for like.

*From:* slide *The silhouette prefers the wrong answer*; handout section 3.7.

### Your prediction, before you run the next cell

k-means scored -0.046 against the truth and DBSCAN 0.941. Which one will the silhouette, which never sees the truth, prefer?

In [13]:
kept = db.labels_ != -1
print(f"sessions DBSCAN placed: {kept.sum()}")
print(f"k-means, all 1,500:          silhouette {silhouette_score(Xs_sess, km2.labels_):.3f}")
print(f"k-means, the same {kept.sum()}:     silhouette {silhouette_score(Xs_sess[kept], km2.labels_[kept]):.3f}")
print(f"DBSCAN,  the {kept.sum()} it placed: silhouette {silhouette_score(Xs_sess[kept], db.labels_[kept]):.3f}")
per_point = silhouette_samples(Xs_sess[kept], db.labels_[kept])
ring = db.labels_[kept] != db.labels_[is_bot][0]           # DBSCAN's other group: the ring
print(f"ring sessions with a negative silhouette: {(per_point[ring] < 0).mean():.1%}")

sessions DBSCAN placed: 1487


k-means, all 1,500:          silhouette 0.401
k-means, the same 1487:     silhouette 0.402


DBSCAN,  the 1487 it placed: silhouette -0.106
ring sessions with a negative silhouette: 99.2%


**What to look at.** On the 1,487 sessions DBSCAN placed, k-means scores 0.402, almost the 0.401 it scores on all 1,500, and DBSCAN -0.106: the silhouette prefers the useless split either way. 99.2% of the ring's sessions have a negative silhouette: on average they are closer to the core in the middle than to the rest of their own ring, most of which lies across the circle.

**Try changing.** score the true grouping, `silhouette_score(Xs_sess, is_bot)`: is the truth itself compact?

## 13. The widest shadow on two columns: $1 + r$, three ways

Annual spend and basket value, standardised. The variance of the shadows on the direction at angle θ is $1 + r \sin 2θ$. Measure it, take the eigenvalues of the correlation matrix, and ask `PCA`.

*From:* slides *Eight columns, not eight independent numbers*; *The direction with the widest shadow*; *The spread of the shadows*; *Why the peak is 1 + r*; *Any number of columns: stated, not derived*; handout sections 4.2 and 4.3.

### Your prediction, before you run the next cell

With r = 0.859, what is the widest spread, and at what angle? What share of the total variance will the first principal component carry?

In [14]:
pair = Xs_acc[:, :2]
r = (pair[:, 0] * pair[:, 1]).mean()          # standardised: the mean product is the correlation
for deg in (0, 45, 135):
    u = np.array([np.cos(np.radians(deg)), np.sin(np.radians(deg))])
    print(f"{deg:>3} degrees: measured variance {(pair @ u).var():.3f}   formula 1 + r sin 2theta = {1 + r * np.sin(2 * np.radians(deg)):.3f}")
print("eigenvalues of [[1, r], [r, 1]]:", np.linalg.eigvalsh([[1, r], [r, 1]])[::-1].round(3))
pca = PCA().fit(pair)
print("PCA components_ (rows):", pca.components_.round(4).tolist())
print("PCA explained_variance_ratio_:", pca.explained_variance_ratio_.round(4), f"  (1 + r) / 2 = {(1 + r) / 2:.4f}")

  0 degrees: measured variance 1.000   formula 1 + r sin 2theta = 1.000
 45 degrees: measured variance 1.859   formula 1 + r sin 2theta = 1.859
135 degrees: measured variance 0.141   formula 1 + r sin 2theta = 0.141
eigenvalues of [[1, r], [r, 1]]: [1.859 0.141]


PCA components_ (rows): [[0.7071, 0.7071], [-0.7071, 0.7071]]
PCA explained_variance_ratio_: [0.9297 0.0703]   (1 + r) / 2 = 0.9297


**What to look at.** Three routes, one answer. The shadows spread 1.000 on the spend axis, 1.859 at 45 degrees and 0.141 at 135, exactly 1 + r sin 2θ with r = 0.859. The eigenvalues of the correlation matrix are the same 1.859 and 0.141, and `PCA` finds the directions (0.7071, 0.7071) and (-0.7071, 0.7071), the diagonals at 45 and 135 degrees, with shares 0.9297 = (1 + r) / 2 and 0.0703.

**Try changing.** do the same with basket value and return rate, `Xs_acc[:, [1, 7]]`: is the widest direction still at 45 degrees, and why?

## 14. Eight columns: the scree, and what a component is made of

PCA on all eight standardised columns. Read the shares of variance, then the first component's weights: a principal component is a new direction mixing the columns, not one of them chosen.

*From:* slides *How many components: the scree plot*; *Any number of columns: stated, not derived*; handout section 4.4.

### Your prediction, before you run the next cell

How many components before the shares drop? Will the first component be annual spend alone?

In [15]:
pca8 = PCA().fit(Xs_acc)
ratio = pca8.explained_variance_ratio_
print("shares:    ", ratio.round(3))
print("cumulative:", ratio.cumsum().round(3))
print("\nfirst component's weight on each column:")
print(pd.Series(pca8.components_[0], index=columns).round(3).to_string())

shares:     [0.442 0.321 0.172 0.024 0.013 0.012 0.01  0.005]
cumulative: [0.442 0.763 0.935 0.96  0.973 0.985 0.995 1.   ]

first component's weight on each column:
annual_spend_eur            0.135
avg_basket_value_eur        0.140
purchase_frequency          0.493
discount_usage_rate        -0.090
days_since_last_purchase   -0.471
weekend_purchase_ratio      0.491
mobile_session_count        0.496
return_rate                -0.039


**What to look at.** The shares drop after the third: 0.442, 0.321, 0.172, then 0.024 and less, and the first three carry 0.935 together; the remaining 6.5% is each column's own noise. The first component is not one column: it weighs purchase frequency 0.493, weekend ratio 0.491, mobile sessions 0.496 and days since the last purchase -0.471, the columns the engagement tendency drives, with little on spend.

**Try changing.** print `pca8.components_[1]`: which hidden tendency does the second component follow?

## 15. Rebuild from the shadow: reconstruction error by hand, then 37 of 40

Rebuilding an account from its shadows is `inverse_transform(transform(x))`: the mean plus the shadows times the kept directions. Do it by hand with three components, check against scikit-learn, and flag the 2% worst rebuilt.

*From:* slides *Fraud that looks ordinary on every column*; *Rebuild each account from its shadow*; *The reconstruction error, as a formula*; *37 of 40*; handout sections 5.1 to 5.3.

### Your prediction, before you run the next cell

Every anomaly's values are ordinary column by column. Will their reconstruction errors still be larger than the genuine accounts'? How many of the 40 flagged will be anomalies?

In [16]:
pca3 = PCA(n_components=3).fit(Xs_acc)
V = pca3.components_                               # 3 directions, each of 8 weights
shadows = (Xs_acc - pca3.mean_) @ V.T              # 3 numbers per account
by_hand = pca3.mean_ + shadows @ V
print(f"largest gap, by hand against inverse_transform: {np.abs(by_hand - pca3.inverse_transform(pca3.transform(Xs_acc))).max():.1e}")

error = ((Xs_acc - by_hand) ** 2).sum(axis=1)
print(f"mean error: anomalies {error[is_anomaly].mean():.3f}, genuine {error[~is_anomaly].mean():.3f}")
flagged = error > np.quantile(error, 0.98)
print(f"flagged {flagged.sum()}, of which anomalies {(flagged & is_anomaly).sum()}")

largest gap, by hand against inverse_transform: 0.0e+00
mean error: anomalies 4.676, genuine 0.431
flagged 40, of which anomalies 37


**What to look at.** Rebuilding by hand, the mean plus the three shadows times the three directions, is exactly `inverse_transform(transform(x))`: the largest gap is 0. The mean reconstruction error is 4.676 for anomalies against 0.431 for genuine accounts, although every anomaly value is ordinary column by column, and 37 of the 40 accounts flagged at the 2% threshold are anomalies.

**Try changing.** keep 2 components and then 6: how many anomalies among the 40 flagged each time?

## 16. Precision and recall move apart, and the 2-D picture points the wrong way

Flag 1%, 2% and 5% of the accounts by reconstruction error and compute precision (the share of flagged that are fraud) and recall (the share of fraud that is flagged). Then measure where the anomalies sit on PCA's 2-D picture.

*From:* slides *37 of 40*; *37 is one batch*; *Why not just plot it and look?*; handout sections 5.3 and 5.4.

### Your prediction, before you run the next cell

At 2%, 40 flagged and 40 planted: are precision and recall equal? At 5%? On the 2-D picture, are the anomalies farther from the middle than genuine accounts, or nearer?

In [17]:
for share in (0.01, 0.02, 0.05):
    hit = error > np.quantile(error, 1 - share)
    caught = (hit & is_anomaly).sum()
    print(f"flag {share:.0%}: {hit.sum():>3} flagged, {caught} anomalies   "
          f"precision {caught}/{hit.sum()} = {caught / hit.sum():.3f}   recall {caught}/{is_anomaly.sum()} = {caught / is_anomaly.sum():.3f}")

picture = PCA(n_components=2).fit_transform(Xs_acc)
middle = picture[~is_anomaly].mean(axis=0)
far = np.linalg.norm(picture - middle, axis=1)
print(f"\n2-D PCA picture, mean distance from the middle: anomalies {far[is_anomaly].mean():.2f}, genuine {far[~is_anomaly].mean():.2f}")
edge = far > np.quantile(far, 0.98)
print(f"flagging the 2% at the edge of the picture instead: {(edge & is_anomaly).sum()} anomalies among {edge.sum()}")

flag 1%:  20 flagged, 20 anomalies   precision 20/20 = 1.000   recall 20/40 = 0.500
flag 2%:  40 flagged, 37 anomalies   precision 37/40 = 0.925   recall 37/40 = 0.925
flag 5%: 100 flagged, 39 anomalies   precision 39/100 = 0.390   recall 39/40 = 0.975

2-D PCA picture, mean distance from the middle: anomalies 1.30, genuine 2.20
flagging the 2% at the edge of the picture instead: 0 anomalies among 40


**What to look at.** At 2%, 40 flagged and 40 planted, so precision and recall are both 37/40 = 0.925, by coincidence. At 1% every flag is right, precision 1.000, but half the fraud is missed, recall 0.500; at 5% recall rises to 0.975 and precision falls to 0.390. Flagging 2% is a budget, not a probability. On the 2-D PCA picture the anomalies sit nearer the middle, 1.30 against 2.20, and flagging the 2% at its edge catches 0 of them.

**Try changing.** flag the edge of the t-SNE picture instead: `TSNE(n_components=2, perplexity=30, random_state=0, init="pca")`.